# TP MLOps II — Limpieza del dataset

Continuación del EDA (`01_dataset.ipynb`). Acá se aplican las decisiones de limpieza que surgieron del análisis exploratorio, para dejar un dataset único, horario, sin nulos y listo para feature engineering / modelado.

**Resumen de lo que se hace:**
1. Cargar `energy` y `weather` crudos.
2. Dropear columnas 100% vacías en `energy`.
3. Interpolar los nulos puntuales restantes (serie horaria, huecos aislados).
4. Resolver duplicados en `weather` (misma hora-ciudad, distinto evento climático categórico).
5. Convertir temperatura de Kelvin a Celsius y pivotear `weather` a formato ancho (una columna de temperatura por ciudad).
6. Unir `energy` + `weather` en un único DataFrame horario.
7. Agregar features de calendario básicas (hora, día de semana, mes, fin de semana).
8. Guardar el resultado en `data/processed/` como parquet.

In [1]:
import pandas as pd
import numpy as np
import pathlib as path

data_path = path.Path('../data')
raw_data_path = data_path / 'raw'

energy = pd.read_csv(raw_data_path / 'energy_dataset.csv')
weather = pd.read_csv(raw_data_path / 'weather_features.csv')

energy['time'] = pd.to_datetime(energy['time'], utc=True)
energy = energy.set_index('time').sort_index()


## 1. Columnas vacías

Del EDA: `generation hydro pumped storage aggregated` y `forecast wind offshore eday ahead` están 100% nulas en todo el dataset. Se confirma y se descartan.

In [2]:
[c for c in energy.columns if 'pumped storage aggregated' in c or 'offshore eday' in c]


['generation hydro pumped storage aggregated',
 'forecast wind offshore eday ahead']

In [3]:
energy = energy.drop(columns=['generation hydro pumped storage aggregated', 'forecast wind offshore eday ahead'])
energy.shape

(35064, 26)

## 2. Nulos puntuales en `energy`

El resto de las columnas de generación (y el target `total load actual`) tenían nulos aislados (~18-36 filas cada una), esparcidos en el tiempo sin formar tramos largos (visto en el EDA). Al ser una serie horaria continua, se interpolan linealmente en el tiempo en vez de dropear filas — así no se rompe la continuidad horaria, algo importante para más adelante cuando se armen features de lags/ventanas móviles.

In [4]:
numeric_cols = energy.select_dtypes(include='number').columns # Select all numeric columns for interpolation
energy[numeric_cols] = energy[numeric_cols].interpolate(method='time') # Interpolate missing values in numeric columns using time-based method


In [5]:
energy.isnull().sum().sum()

0

## 3. Duplicados en `weather`

`weather` tenía 178.396 filas en vez de las 175.320 esperadas (35.064 horas × 5 ciudades) — 3.076 de más.

In [6]:
weather['dt_iso'] = pd.to_datetime(weather['dt_iso'])
weather.duplicated(subset=['dt_iso', 'city_name']).sum()


C:\Users\alanv\AppData\Local\Temp\ipykernel_27468\39494601.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  weather['dt_iso'] = pd.to_datetime(weather['dt_iso'])


3076

Se inspeccionan esos duplicados: **no son filas idénticas**. Comparten timestamp, ciudad y todas las variables numéricas (`temp`, `humidity`, `pressure`, etc.), pero difieren en el evento climático categórico reportado (`weather_id`, `weather_main`, `weather_description`, `weather_icon`) — por ejemplo, la misma hora aparece una vez como "rain" y otra como "thunderstorm". Es un artefacto de cómo la API de OpenWeather emite múltiples eventos para una misma hora.

In [7]:
dup_mask = weather.duplicated(subset=['dt_iso', 'city_name'], keep=False)
weather[dup_mask].sort_values(['city_name', 'dt_iso']).head(20)


,dt_iso,city_name,temp,temp_min,temp_max,pressure,humidity,wind_speed,wind_deg,rain_1h,rain_3h,snow_3h,clouds_all,weather_id,weather_main,weather_description,weather_icon
109256,2015-03-20 21:00:00+01:00,Barcelona,286.81,282.59,291.48,1013,76,12,50,0.3,0.0,0.0,40,500,rain,light rain,10n
109257,2015-03-20 21:00:00+01:00,Barcelona,286.81,282.59,291.48,1013,76,12,50,0.3,0.0,0.0,40,301,drizzle,drizzle,09n
111170,2015-06-08 15:00:00+02:00,Barcelona,299.47,291.48,304.82,1017,57,3,180,12.0,0.0,0.0,20,503,rain,very heavy rain,10d
111171,2015-06-08 15:00:00+02:00,Barcelona,299.47,291.48,304.82,1017,57,3,180,12.0,0.0,0.0,20,211,thunderstorm,thunderstorm,11d
111172,2015-06-08 16:00:00+02:00,Barcelona,297.59,292.04,300.37,1017,54,2,170,3.0,0.0,0.0,20,502,rain,heavy intensity rain,10d
111173,2015-06-08 16:00:00+02:00,Barcelona,297.59,292.04,300.37,1017,54,2,170,3.0,0.0,0.0,20,211,thunderstorm,thunderstorm,11d
111174,2015-06-08 17:00:00+02:00,Barcelona,297.03,292.04,300.15,1017,54,6,60,3.0,0.0,0.0,20,502,rain,heavy intensity rain,10d
111175,2015-06-08 17:00:00+02:00,Barcelona,297.03,292.04,300.15,1017,54,6,60,3.0,0.0,0.0,20,211,thunderstorm,thunderstorm,11d
111339,2015-06-15 13:00:00+02:00,Barcelona,294.73,289.26,298.15,1016,64,4,130,3.0,0.0,0.0,40,502,rain,heavy intensity rain,10d
111340,2015-06-15 13:00:00+02:00,Barcelona,294.73,289.26,298.15,1016,64,4,130,3.0,0.0,0.0,40,211,thunderstorm,thunderstorm,11d


Como el objetivo es forecasting de demanda (donde `temp` es la variable climática relevante, no el detalle categórico del clima), alcanza con quedarse con **una fila por hora-ciudad** — se descarta el resto, ya que las variables numéricas que importan no cambian entre duplicados.

En la misma celda se resuelve el resto de la preparación de `weather`:
- Conversión de temperatura de **Kelvin a Celsius** (más interpretable).
- Pivoteo a formato ancho: de "una fila por hora-ciudad" a "una fila por hora, con una columna `temp_<ciudad>` por cada una de las 5 ciudades", para poder unirlo con `energy` (que es nacional, una fila por hora).

In [8]:
weather = weather.drop_duplicates(subset=['dt_iso', 'city_name'], keep='first')
weather.shape

# Convert temperatures from Kelvin to Celsius
for col in ['temp', 'temp_min', 'temp_max']:
    weather[col] = weather[col] - 273.15
    
weather_wide = weather.pivot_table(index='dt_iso', columns='city_name', values='temp') # Pivot the weather data to have dates as rows and cities as columns
weather_wide.columns = [f'temp_{c.strip()}' for c in weather_wide.columns] # Rename columns to include 'temp_' prefix for clarity
# weather_wide.head()
weather_wide.shape
weather_wide.columns.tolist()

['temp_Barcelona',
 'temp_Bilbao',
 'temp_Madrid',
 'temp_Seville',
 'temp_Valencia']

## 4. Unión de `energy` + `weather` y features de calendario

Se unen ambos DataFrames por el índice horario (`join`, ambos quedan alineados por ser series horarias completas sin huecos). Se agregan además las features de calendario que ya se habían usado exploratoriamente en el EDA (hora, día de semana, mes, flag de fin de semana), ahora como parte fija del dataset limpio.

In [9]:
df = energy.join(weather_wide, how='left')
df.shape
# df.isnull().sum().sum()

df.index = pd.to_datetime(df.index, utc=True) # Ensure the index is in datetime format with UTC timezone
df['hour'] = df.index.hour
df['dow'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = (df['dow'] >= 5).astype(int)

## 5. Guardado

Se persiste el resultado en `data/processed/energy_weather_clean.parquet`. Se usa **parquet** en vez de CSV porque preserva tipos de dato y el índice datetime (con timezone) sin ambigüedad, y es más liviano/rápido de leer en los siguientes notebooks.

In [10]:
processed_path = data_path / 'processed'
processed_path.mkdir(exist_ok=True)
df.to_parquet(processed_path / 'energy_weather_clean.parquet')


## Resumen

Dataset limpio: **35.064 filas × 31 columnas**, horario, sin nulos, con:
- Generación eléctrica por fuente, forecasts oficiales, precios (de `energy`, columnas vacías descartadas).
- Temperatura por ciudad (`temp_Madrid`, `temp_Barcelona`, `temp_Valencia`, `temp_Seville`, `temp_Bilbao`), en Celsius.
- Features de calendario: `hour`, `dow`, `month`, `is_weekend`.

**Próximo paso:** split temporal (train/test) y un primer baseline de forecasting para `total load actual`.